# Scraping Component
This notebook automatically discovers and scrapes all South Asian cuisine related pages
from Wikipedia and saves them as a JSON corpus for use in the CuisineRAG system.

**Output:** `data/raw/south_asian_corpus.json`

## 1. Install & Import Libraries

In [7]:
import wikipediaapi
import json
import os
from pathlib import Path
import re
import time
import numpy as np
from urllib.parse import unquote
from sentence_transformers import SentenceTransformer

# Initialize Wikipedia API
wiki = wikipediaapi.Wikipedia(
    language='en',
    user_agent='CuisineRAG/1.0 (mingyi.jin@student.manchester.ac.uk)'
)

print('Libraries loaded successfully!')

from enum import Enum

class Link_type(Enum):
    WIKIPEDIA = 1
    WIKIBOOK = 2
    CUISINES80 = 3

from dataclasses import dataclass, asdict

@dataclass
class Section:
    section_title: str
    section_text: str


Libraries loaded successfully!


## 2. Discover All South Asian Food Pages
We start from the main South Asian cuisine Wikipedia page and extract
all linked dish/food pages automatically.

In [8]:
# Entry point pages for South Asian cuisine
SEED_PAGES = [
    # Regional cuisine overviews
    'South_Asian_cuisine',
    'Indian_cuisine',
    'Pakistani_cuisine',
    'Bangladeshi_cuisine',
    'Sri_Lankan_cuisine',
    'Nepalese_cuisine',
    'Afghan_cuisine',
    'Maldivian_cuisine',
    'Bhutanese_cuisine',
    # High-link-density dish pages
    'Biryani',
    'Curry',
    'Kebab',
    'Samosa',
    'Dosa',
    'Tandoor',
    'Haleem',
    'Kabuli_palaw',
]

EXCLUDE_TITLE_KEYWORDS = {
    'template', 'list of', 'index of', 'disambiguation', 'category',
    'outline of', 'timeline of', 'history of', 'geography of', 'demographics'
}

FOOD_KEYWORDS = [
    'cuisine', 'food', 'dish', 'recipe', 'cooking', 'curry', 'rice', 'bread',
    'spice', 'biryani', 'masala', 'dal', 'roti', 'naan', 'chutney', 'kebab',
    'samosa', 'dessert', 'sweet', 'snack', 'drink', 'beverage', 'soup',
    'salad', 'chicken', 'lamb', 'mutton', 'lentil', 'vegetable', 'paneer'
]

SOUTH_ASIA_KEYWORDS = [
    'south asia', 'india', 'indian', 'pakistan', 'pakistani',
    'bangladesh', 'bangladeshi', 'sri lanka', 'sri lankan',
    'nepal', 'nepalese', 'kashmir', 'bengal', 'punjab',
    'tamil', 'mughlai', 'awadhi', 'hyderabadi'
]


def normalize_page_title(title: str) -> str:
    """Normalize wikipedia page keys into comparable title strings."""
    cleaned = unquote(title).replace('/wiki/', '').replace('_', ' ').strip()
    cleaned = re.sub(r'\s+', ' ', cleaned)
    return cleaned


def is_food_title_candidate(page_title: str) -> bool:
    """Title-level filter to remove obvious non-food/non-article pages."""
    title = normalize_page_title(page_title).lower()
    if ':' in title:
        return False
    if any(bad in title for bad in EXCLUDE_TITLE_KEYWORDS):
        return False
    return any(k in title for k in FOOD_KEYWORDS)


def discover_links_from_seed(seed_title: str) -> set[str]:
    """Discover linked pages from a seed article via wikipediaapi links."""
    page = wiki.page(seed_title)
    if not page.exists():
        print(f'  ! Seed missing: {seed_title}')
        return set()

    discovered = set()
    for linked_title in page.links.keys():
        if is_food_title_candidate(linked_title):
            discovered.add(linked_title)
    return discovered


print('Discovering pages from seed pages...')
all_candidate_pages = set()
discovery_log = []

for seed in SEED_PAGES:
    print(f'  Scanning: {seed}')
    links = discover_links_from_seed(seed)
    all_candidate_pages.update(links)
    discovery_log.append({'seed': seed, 'discovered_candidates': len(links)})
    time.sleep(0.5)

print(f'\nTotal candidate pages found: {len(all_candidate_pages)}')


Discovering pages from seed pages...
  Scanning: South_Asian_cuisine
  Scanning: Indian_cuisine
  Scanning: Pakistani_cuisine
  Scanning: Bangladeshi_cuisine
  Scanning: Sri_Lankan_cuisine
  Scanning: Nepalese_cuisine
  Scanning: Afghan_cuisine
  Scanning: Maldivian_cuisine
  Scanning: Bhutanese_cuisine
  Scanning: Biryani
  Scanning: Curry
  Scanning: Kebab
  Scanning: Samosa
  Scanning: Dosa
  Scanning: Tandoor
  Scanning: Haleem
  Scanning: Kabuli_palaw

Total candidate pages found: 1138


## 3. Filter Relevant Pages
Not all links are food related. We filter by checking if the page
title or content contains food/cuisine related keywords.

## 3a. Scope Prototype Test
Use this cell to inspect positive-vs-negative prototype scores for sample cuisine titles.


In [9]:
theme_test_samples = [
    'British cuisine',
    'Pakistani cuisine',
    'Kerala cuisine',
    'Naan',
    'Mexican cuisine',
    'Tamil cuisine',
]

theme_test_positive = [
    'Pakistani cuisine',
    'Bangladeshi cuisine',
    'Nepalese cuisine',
    'Sri Lankan cuisine',
    'Kerala cuisine',
    'Tamil cuisine',
    'Bengali cuisine',
    'Andhra cuisine',
    'Goan cuisine',
    'Kashmiri cuisine',
    'Newar cuisine',
    'Sindhi cuisine',
    'Mughlai cuisine',
    'Hyderabadi biryani',
    'Tibetan cuisine',
    'Maldivian cuisine',
    'Naan',
]
theme_test_negative = [
    'British cuisine',
    'Mexican cuisine',
    'French cuisine',
    'Japanese cuisine',
    'Italian cuisine',
    'Greek cuisine',
    'Tacos',
    'Sushi',
    'Baguette',
    'Paella',
]
theme_test_model_name = 'BAAI/bge-base-en-v1.5'

theme_test_model = SentenceTransformer(theme_test_model_name)
positive_prototypes = theme_test_model.encode(
    theme_test_positive,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
negative_prototypes = theme_test_model.encode(
    theme_test_negative,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print(f'Scope prototype test ({theme_test_model_name}):')
for sample in theme_test_samples:
    sample_embedding = theme_test_model.encode(
        [sample],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0]
    positive_score = float(np.dot(positive_prototypes, sample_embedding).mean())
    negative_score = float(np.dot(negative_prototypes, sample_embedding).mean())
    margin = positive_score - negative_score
    print(f'{sample:<25} -> positive={positive_score:.4f} | negative={negative_score:.4f} | margin={margin:.4f}')


Scope prototype test (BAAI/bge-base-en-v1.5):
British cuisine           -> positive=0.6097 | negative=0.6601 | margin=-0.0505
Pakistani cuisine         -> positive=0.6913 | negative=0.6063 | margin=0.0850
Kerala cuisine            -> positive=0.6690 | negative=0.6058 | margin=0.0632
Naan                      -> positive=0.4949 | negative=0.3938 | margin=0.1011
Mexican cuisine           -> positive=0.6004 | negative=0.6653 | margin=-0.0649
Tamil cuisine             -> positive=0.6877 | negative=0.5844 | margin=0.1033


In [10]:
def clean_categories(raw_categories: list[str]) -> list[str]:
    """Remove hidden or maintenance categories using keyword rules."""
    noise_patterns = [
        'hidden categories',
        'all articles',
        'articles ',
        'wikipedia ',
        'cs1',
        'short description',
        'wikidata',
        'webarchive',
        'wayback',
        'pages using',
        'commons category link',
        'use dmy dates',
        'use mdy dates',
        'use british english',
        'use american english',
        'use indian english',
        'good articles',
        'featured articles',
        'all pages needing cleanup',
        'cleanup',
        'stub articles',
        'stubs',
        'region topic template using suffix',
    ]
    cleaned = []
    for c in raw_categories:
        lc = c.lower().strip()
        if any(p in lc for p in noise_patterns):
            continue
        cleaned.append(c.replace('Category:', '').strip())
    return cleaned


GENERIC_OUT_OF_SCOPE_TITLES = {
    'cuisine', 'global cuisine', 'lists of prepared foods', 'food and drink',
    'mixed rice dish', 'brain as food', 'chicken as food', 'duck as food', 'eggs as food',
    'food', 'cooking', 'drink', 'dessert', 'salad', 'soup', 'snack', 'street food',
    'fast food', 'regional cuisine', 'vegetarian cuisine'
}

SOUTH_ASIA_WHITELIST_KEYWORDS = [
    'india', 'indian', 'pakistan', 'pakistani', 'bangladesh', 'bangladeshi',
    'sri lanka', 'sri lankan', 'nepal', 'nepalese', 'bhutan', 'bhutanese',
    'tibet', 'tibetan', 'kashmir', 'kashmiri', 'bengal', 'bengali', 'punjab', 'punjabi',
    'tamil', 'andhra', 'kerala', 'bihari', 'jharkhandi', 'chhattisgarhi', 'karnataka',
    'kumaoni', 'newar', 'sindhi', 'mughlai', 'hyderabadi', 'goan', 'jaffna', 'garo',
    'hazara', 'arunachali', 'maldivian', 'northeast india',
    'naan', 'roti', 'biryani', 'dal', 'achar', 'thukpa', 'momo', 'momos',
    'makki ki roti', 'gothamba roti', 'pol roti', 'kadala', 'wai wai',
    'puffed rice', 'flattened rice', 'red rice',
]

EMBEDDING_MODEL_NAME = 'BAAI/bge-base-en-v1.5'
POSITIVE_SCOPE_PROTOTYPES = [
    'Pakistani cuisine',
    'Bangladeshi cuisine',
    'Nepalese cuisine',
    'Sri Lankan cuisine',
    'Kerala cuisine',
    'Tamil cuisine',
    'Bengali cuisine',
    'Andhra cuisine',
    'Goan cuisine',
    'Kashmiri cuisine',
    'Newar cuisine',
    'Sindhi cuisine',
    'Mughlai cuisine',
    'Hyderabadi biryani',
    'Tibetan cuisine',
    'Maldivian cuisine',
    'Naan',
]
NEGATIVE_SCOPE_PROTOTYPES = [
    'British cuisine',
    'Mexican cuisine',
    'French cuisine',
    'Japanese cuisine',
    'Italian cuisine',
    'Greek cuisine',
    'Tacos',
    'Sushi',
    'Baguette',
    'Paella',
]
SCOPE_MARGIN_THRESHOLD = 0.04

_scope_model = None
_scope_prototype_embeddings = None

# Fields used only during filtering — stripped before saving to corpus
_EPHEMERAL_FIELDS = {'num_characters', 'food_density', 'scope_positive_score', 'scope_negative_score', 'scope_margin'}


def is_out_of_scope_title(title: str) -> bool:
    t = title.lower().strip().replace('_', ' ')
    return t in GENERIC_OUT_OF_SCOPE_TITLES


def passes_whitelist_scope(title: str, summary: str, categories: list[str]) -> bool:
    """Check only title + summary + categories to avoid false positives from body text."""
    combined = f"{title}\n{summary}\n{', '.join(categories[:8])}".lower()
    return any(k in combined for k in SOUTH_ASIA_WHITELIST_KEYWORDS)


def get_scope_model() -> SentenceTransformer:
    global _scope_model
    if _scope_model is None:
        _scope_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    return _scope_model


def get_scope_prototype_embeddings() -> tuple[np.ndarray, np.ndarray]:
    global _scope_prototype_embeddings
    if _scope_prototype_embeddings is None:
        model = get_scope_model()
        positive_embeddings = model.encode(
            POSITIVE_SCOPE_PROTOTYPES,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        negative_embeddings = model.encode(
            NEGATIVE_SCOPE_PROTOTYPES,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        _scope_prototype_embeddings = (positive_embeddings, negative_embeddings)
    return _scope_prototype_embeddings


def build_scope_text(title: str, summary: str, categories: list[str], text: str) -> str:
    category_text = ', '.join(categories[:8])
    return f"Title: {title}\nSummary: {summary}\nCategories: {category_text}\nBody: {text[:1200]}"


def compute_scope_scores(title: str, summary: str, categories: list[str], text: str) -> tuple[float, float, float]:
    model = get_scope_model()
    doc_text = build_scope_text(title, summary, categories, text)
    doc_embedding = model.encode(
        [doc_text],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0]
    positive_embeddings, negative_embeddings = get_scope_prototype_embeddings()
    positive_score = float(np.dot(positive_embeddings, doc_embedding).mean())
    negative_score = float(np.dot(negative_embeddings, doc_embedding).mean())
    margin = positive_score - negative_score
    return positive_score, negative_score, margin


def scrape_wikipedia_page(page_title: str) -> dict | None:
    """
    Scrape a Wikipedia page and return structured data in wikibook-compatible format.
    Returns None for missing pages.
    """
    page = wiki.page(page_title)
    if not page.exists():
        return None

    text = f'# {page.title}\n\n'
    text += (page.summary or '') + '\n\n'

    # Build content sections — summary is stored separately, so skip the
    # duplicate 'introduction' entry that would mirror page.summary.
    content = []

    for section in page.sections:
        section_body = section.text or ''
        text += f'## {section.title}\n'
        text += section_body + '\n\n'
        if section_body.strip():
            content.append(asdict(Section(section_title=section.title, section_text=section_body)))

        for subsection in section.sections:
            subsection_body = subsection.text or ''
            text += f'### {subsection.title}\n'
            text += subsection_body + '\n\n'
            if subsection_body.strip():
                content.append(
                    asdict(Section(section_title=f'{section.title} / {subsection.title}', section_text=subsection_body))
                )

    raw_categories = sorted(page.categories.keys()) if getattr(page, 'categories', None) else []
    categories = clean_categories(raw_categories)

    return {
        'title':    page.title,
        'url':      page.fullurl,
        'type':     'WIKIPEDIA',
        'categories': categories,
        'summary':  {
            'section_title': 'Introduction',
            'section_text':  page.summary or '',
        },
        'content':  content,
        'num_characters': len(text),  # kept temporarily for filtering below
    }


def content_food_density(text: str) -> float:
    """Rough ratio of food-keyword hits in text; used as content-level filter."""
    lowered = text.lower()
    hits = sum(lowered.count(k) for k in FOOD_KEYWORDS)
    token_count = max(1, len(lowered.split()))
    return hits / token_count


def passes_keyword_scope(title: str, summary: str, text: str) -> bool:
    """Fast rule-based filter for obvious South Asia relevance."""
    combined = f"{title}\n{summary}\n{text[:1600]}".lower()
    return any(k in combined for k in SOUTH_ASIA_KEYWORDS)


MIN_CHARS = 400
MIN_FOOD_DENSITY = 0.003
MAX_ACCEPTED_PAGES = None  # set integer for sampling, None for full crawl

corpus = []
seen_urls = set()
crawl_log = {
    'config': {
        'seeds': SEED_PAGES,
        'min_chars': MIN_CHARS,
        'min_food_density': MIN_FOOD_DENSITY,
        'south_asia_keywords': SOUTH_ASIA_KEYWORDS,
        'embedding_model_name': EMBEDDING_MODEL_NAME,
        'scope_margin_threshold': SCOPE_MARGIN_THRESHOLD,
        'positive_scope_prototypes': POSITIVE_SCOPE_PROTOTYPES,
        'negative_scope_prototypes': NEGATIVE_SCOPE_PROTOTYPES,
        'south_asia_whitelist_keywords': SOUTH_ASIA_WHITELIST_KEYWORDS,
    },
    'discovery': discovery_log,
    'accepted': [],
    'skipped': [],
    'errors': [],
}

food_pages = sorted(all_candidate_pages)
print(f'Scraping {len(food_pages)} candidate pages...\n')

for i, page_title in enumerate(food_pages, start=1):
    if MAX_ACCEPTED_PAGES is not None and len(corpus) >= MAX_ACCEPTED_PAGES:
        print(f"Reached MAX_ACCEPTED_PAGES={MAX_ACCEPTED_PAGES}. Stopping early.")
        break

    try:
        result = scrape_wikipedia_page(page_title)
        if not result:
            crawl_log['skipped'].append({'title': page_title, 'reason': 'page_not_found'})
            print(f'[{i}/{len(food_pages)}] - Skipped: {page_title} (not found)')
            continue

        summary_text = result['summary']['section_text']
        text_for_filter = '\n\n'.join(
            [summary_text] + [s.get('section_text', '') for s in result.get('content', [])]
        )
        density = content_food_density(text_for_filter)
        result['food_density'] = round(density, 6)

        if result['url'] in seen_urls:
            crawl_log['skipped'].append({'title': result['title'], 'reason': 'duplicate_url', 'url': result['url']})
            print(f'[{i}/{len(food_pages)}] - Skipped: {result["title"]} (duplicate URL)')
            continue
        elif result['num_characters'] < MIN_CHARS:
            crawl_log['skipped'].append({'title': result['title'], 'reason': 'too_short', 'num_characters': result['num_characters']})
            print(f'[{i}/{len(food_pages)}] - Skipped: {result["title"]} (too short)')
            continue
        elif density < MIN_FOOD_DENSITY:
            crawl_log['skipped'].append({'title': result['title'], 'reason': 'low_food_density', 'food_density': round(density, 6)})
            print(f'[{i}/{len(food_pages)}] - Skipped: {result["title"]} (low food density {density:.4f})')
            continue
        elif is_out_of_scope_title(result['title']):
            crawl_log['skipped'].append({'title': result['title'], 'reason': 'out_of_scope_title'})
            print(f'[{i}/{len(food_pages)}] - Skipped: {result["title"]} (generic out-of-scope title)')
            continue
        elif passes_whitelist_scope(result['title'], summary_text, result.get('categories', [])):
            scope_margin = 1.0
        elif not passes_keyword_scope(result['title'], summary_text, text_for_filter):
            crawl_log['skipped'].append({'title': result['title'], 'reason': 'keyword_scope_reject'})
            print(f'[{i}/{len(food_pages)}] - Skipped: {result["title"]} (keyword scope reject)')
            continue
        else:
            positive_score, negative_score, scope_margin = compute_scope_scores(
                result['title'],
                summary_text,
                result.get('categories', []),
                text_for_filter,
            )
            if scope_margin < SCOPE_MARGIN_THRESHOLD:
                crawl_log['skipped'].append({
                    'title': result['title'],
                    'reason': 'embedding_scope_reject',
                    'scope_margin': round(scope_margin, 4),
                })
                print(
                    f'[{i}/{len(food_pages)}] - Skipped: {result["title"]} '
                    f'(positive={positive_score:.3f}, negative={negative_score:.3f}, margin={scope_margin:.3f})'
                )
                continue

        # Strip ephemeral filter fields before saving
        for field in _EPHEMERAL_FIELDS:
            result.pop(field, None)

        seen_urls.add(result['url'])
        corpus.append(result)
        crawl_log['accepted'].append({
            'title': result['title'],
            'url': result['url'],
            'scope_margin': round(scope_margin, 4),
        })
        print(f'[{i}/{len(food_pages)}] + Accepted: {result["title"]} (margin={scope_margin:.3f})')

    except Exception as e:
        crawl_log['errors'].append({'title': page_title, 'error': str(e)})
        print(f'[{i}/{len(food_pages)}] ! Error: {page_title} -> {e}')

    time.sleep(0.3)

print(f'\nAccepted pages: {len(corpus)}')
print(f'Skipped pages : {len(crawl_log["skipped"])}')
print(f'Errors        : {len(crawl_log["errors"])}')


Scraping 1138 candidate pages...

[1/1138] - Skipped: 3-in-1 (fast food dish) (keyword scope reject)
[2/1138] - Skipped: A Book of Mediterranean Food (keyword scope reject)
[3/1138] - Skipped: Georgian cuisine (keyword scope reject)
[4/1138] - Skipped: Acadian cuisine (keyword scope reject)
[5/1138] + Accepted: Acehnese cuisine (margin=1.000)
[6/1138] - Skipped: Achari paneer tikka (not found)
[7/1138] + Accepted: Ada (food) (margin=1.000)
[8/1138] - Skipped: Adana kebabı (keyword scope reject)
[9/1138] + Accepted: Naan (margin=1.000)
[10/1138] + Accepted: Afghan cuisine (margin=1.000)
[11/1138] - Skipped: Afghan salad (positive=0.539, negative=0.531, margin=0.008)
[12/1138] - Skipped: Afghan cuisine (duplicate URL)
[13/1138] + Accepted: African cuisine (margin=1.000)
[14/1138] - Skipped: Aging (food) (keyword scope reject)
[15/1138] - Skipped: Ainu cuisine (keyword scope reject)
[16/1138] + Accepted: Ajapsandali (margin=1.000)
[17/1138] - Skipped: Akok (food) (keyword scope reject)
[1

## 4. Scrape Each Page
Loop through all discovered food pages and scrape their full text.
Text is formatted with markdown headers for use in chunking.

In [11]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

output_dir = PROJECT_ROOT / 'data/raw'
output_dir.mkdir(parents=True, exist_ok=True)

corpus_path = output_dir / 'south_asian_corpus.json'
log_path = output_dir / 'crawl_log.json'

with corpus_path.open('w', encoding='utf-8') as f:
    json.dump(corpus, f, ensure_ascii=False, indent=2)

with log_path.open('w', encoding='utf-8') as f:
    json.dump(crawl_log, f, ensure_ascii=False, indent=2)

print(f'Corpus saved to: {corpus_path}')
print(f'Crawl log saved to: {log_path}')
print(f'Total pages: {len(corpus)}')


Corpus saved to: /Users/jinmingyi/CuisineRAG/data/raw/south_asian_corpus.json
Crawl log saved to: /Users/jinmingyi/CuisineRAG/data/raw/crawl_log.json
Total pages: 327


## 5. Save Corpus to JSON
Save all scraped pages to `data/raw/south_asian_corpus.json`.
This file will be loaded by `chunking.ipynb`.

In [12]:
print('=== CORPUS SUMMARY ===')
if not corpus:
    print('No pages accepted. Consider lowering thresholds.')
else:
    total_sections = sum(len(p.get('content', [])) for p in corpus)
    print(f'Total pages accepted : {len(corpus)}')
    print(f'Total sections       : {total_sections:,}')
    print(f'Avg sections/page    : {total_sections // len(corpus)}')
    print(f'Largest page         : {max(corpus, key=lambda x: len(x.get("content", [])))["title"]}')

    print('\n=== SAMPLE PAGE TITLES ===')
    for page in corpus[:10]:
        print(f'  - {page["title"]} ({len(page.get("content", []))} sections)')

    print('\n=== SAMPLE SECTION PREVIEW (first page) ===')
    first_content = corpus[0].get('content', [])
    if first_content:
        print(first_content[0].get('section_text', '')[:500])
    else:
        print(corpus[0].get('summary', {}).get('section_text', '')[:500])

print('\n=== CRAWL LOG SUMMARY ===')
print(f"Accepted entries : {len(crawl_log['accepted'])}")
print(f"Skipped entries  : {len(crawl_log['skipped'])}")
print(f"Error entries    : {len(crawl_log['errors'])}")


=== CORPUS SUMMARY ===
Total pages accepted : 327
Total sections       : 2,700
Avg sections/page    : 8
Largest page         : Indian cuisine

=== SAMPLE PAGE TITLES ===
  - Acehnese cuisine (5 sections)
  - Ada (food) (2 sections)
  - Naan (9 sections)
  - Afghan cuisine (19 sections)
  - African cuisine (12 sections)
  - Ajapsandali (1 sections)
  - Amygdalopita (3 sections)
  - Anchovies as food (5 sections)
  - Andalusia (47 sections)
  - Andalusian cuisine (7 sections)

=== SAMPLE SECTION PREVIEW (first page) ===
Asam sunti, a condiment made of star fruit and salt.

=== CRAWL LOG SUMMARY ===
Accepted entries : 327
Skipped entries  : 810
Error entries    : 1


## 6. Preview the Corpus
Quick look at what was scraped.

## 7. Next Step
The corpus is now saved at `data/raw/south_asian_corpus.json`.

Open `chunking.ipynb` and load this file to start chunking:
```python
with open('data/raw/south_asian_corpus.json', 'r') as f:
    corpus = json.load(f)
```